In [1]:
import joblib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PATH = "models/linear_regression.pkl"
DATASET_PATH = "data/processed/modelling_dataset.csv"
RAW_DATA_PATH = "data/raw/crime_data_london_raw.csv"
 
LINEAR_FEATURES = [
    "year", "month_num", "quarter", "month_sin", "month_cos", "season",
    "crime_lag_1", "crime_lag_3", "crime_lag_12",
    "crime_rolling_3", "crime_rolling_6",
    "imd_score",
    "population_2023", "population_density_per_km2",
    "median_annual_earnings_2023", "overcrowding_rate",
]
TREE_FEATURES = [
    "year", "month_num", "quarter", "month_sin", "month_cos", "season",
    "crime_lag_1", "crime_lag_3", "crime_lag_12",
    "crime_rolling_3", "crime_rolling_6",
    "imd_score", "income_deprivation_score", "employment_deprivation_score",
    "crime_deprivation_score",
    "population_2023", "population_density_per_km2",
    "claimant_count_rate_2023", "median_annual_earnings_2023",
    "median_house_price_2023", "overcrowding_rate",
]
FEATURES = LINEAR_FEATURES if "linear" in MODEL_PATH.lower() else TREE_FEATURES
 
SEASON_MAP = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
 

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("  DETAILED CRIME PREDICTION TOOL")
print("=" * 60)
print("\nLoading model and data...")
 
model = joblib.load(MODEL_PATH)
 
df = pd.read_csv(DATASET_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.dropna(subset=["crime_lag_1", "crime_lag_3", "crime_lag_12",
                        "crime_rolling_3", "crime_rolling_6"]).reset_index(drop=True)
 
print("Loading raw crime data (this may take 10-20s due to size)...")
raw_df = pd.read_csv(RAW_DATA_PATH)
print(f"  Modelling data : {len(df):,} rows")
print(f"  Raw crime data : {len(raw_df):,} records")
print(f"  Categories     : {raw_df['category'].nunique()}")
 

  DETAILED CRIME PREDICTION TOOL

Loading model and data...
Loading raw crime data (this may take 10-20s due to size)...
  Modelling data : 792 rows
  Raw crime data : 3,421,509 records
  Categories     : 14


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CORE FORECASTING FUNCTIONS (same as 07_test_models.py)
# ─────────────────────────────────────────────────────────────────────────────
def build_feature_row(history, target_date, borough):
    h = history[history["borough"] == borough].sort_values("date").reset_index(drop=True)
    socio_cols = [
        "imd_score", "income_deprivation_score", "employment_deprivation_score",
        "crime_deprivation_score", "population_2023", "population_density_per_km2",
        "claimant_count_rate_2023", "median_annual_earnings_2023",
        "median_house_price_2023", "overcrowding_rate",
    ]
    socio = h.iloc[-1][socio_cols].to_dict()
    lag_1  = h.iloc[-1]["crime_count"]
    lag_3  = h.iloc[-3]["crime_count"]  if len(h) >= 3  else lag_1
    lag_12 = h.iloc[-12]["crime_count"] if len(h) >= 12 else lag_1
    rolling_3 = h.iloc[-3:]["crime_count"].mean() if len(h) >= 3 else lag_1
    rolling_6 = h.iloc[-6:]["crime_count"].mean() if len(h) >= 6 else lag_1
    month_num = target_date.month
    return pd.DataFrame([{
        "year": target_date.year, "month_num": month_num,
        "quarter": (month_num - 1) // 3 + 1,
        "month_sin": np.sin(2 * np.pi * month_num / 12),
        "month_cos": np.cos(2 * np.pi * month_num / 12),
        "season": SEASON_MAP[month_num],
        "crime_lag_1": lag_1, "crime_lag_3": lag_3, "crime_lag_12": lag_12,
        "crime_rolling_3": rolling_3, "crime_rolling_6": rolling_6,
        **socio,
    }])
 
def forecast_recursive(model, history, borough, num_months):
    h = history.copy()
    forecasts = []
    last_date = h[h["borough"] == borough]["date"].max()
    for _ in range(num_months):
        target_date = last_date + pd.DateOffset(months=1)
        X = build_feature_row(h, target_date, borough)
        pred = float(model.predict(X[FEATURES])[0])
        forecasts.append({"date": target_date, "prediction": pred})
        new_row = X.iloc[0].to_dict()
        new_row["borough"], new_row["date"] = borough, target_date
        new_row["month"] = target_date.strftime("%Y-%m")
        new_row["crime_count"] = pred
        h = pd.concat([h, pd.DataFrame([new_row])], ignore_index=True)
        last_date = target_date
    return forecasts
 

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# DETAIL FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
def get_confidence_interval(prediction):
    """95% interval using residuals from the test period."""
    test_start = sorted(df["date"].unique())[int(len(df["date"].unique()) * 0.8)]
    test_data = df[df["date"] >= test_start]
    y_pred = model.predict(test_data[FEATURES])
    residuals = test_data["crime_count"].values - y_pred
    lower = prediction + np.quantile(residuals, 0.025)
    upper = prediction + np.quantile(residuals, 0.975)
    return max(0, lower), upper
 
def get_yoy_comparison(borough, target_date):
    """Compare to same month last year."""
    last_year = target_date - pd.DateOffset(years=1)
    yoy = df[(df["borough"] == borough) & (df["date"] == last_year)]
    return float(yoy["crime_count"].iloc[0]) if len(yoy) else None
 
def get_trend(borough):
    """Trend over last 6 months of available data."""
    borough_data = df[df["borough"] == borough].sort_values("date")
    recent = borough_data.tail(6)["crime_count"].values
    if len(recent) < 2:
        return "unknown", 0
    pct_change = (recent[-1] - recent[0]) / recent[0] * 100
    if abs(pct_change) < 5:
        return "flat", pct_change
    return ("increasing" if pct_change > 0 else "decreasing"), pct_change
 
def get_borough_rank(predicted_value, target_date):
    """Where does this prediction rank among recent borough crime levels?"""
    latest = df[df["date"] == df["date"].max()].set_index("borough")["crime_count"]
    rank = int((latest > predicted_value).sum()) + 1
    return rank, len(latest)
 
def get_category_breakdown(borough, predicted_total):
    """Apply this borough's recent category proportions to the predicted total."""
    borough_data = raw_df[raw_df["borough"] == borough]
    recent_months = sorted(borough_data["month"].unique())[-12:]
    recent = borough_data[borough_data["month"].isin(recent_months)]
    proportions = recent["category"].value_counts(normalize=True)
    return (proportions * predicted_total).round().astype(int)
 
def get_top_streets(borough, n=5):
    """Top n streets by crime count over last 12 months."""
    borough_data = raw_df[raw_df["borough"] == borough]
    recent_months = sorted(borough_data["month"].unique())[-12:]
    recent = borough_data[borough_data["month"].isin(recent_months)]
    streets = recent["street"].dropna()
    streets = streets[streets.astype(str).str.strip() != ""]
    return streets.value_counts().head(n)
 
def get_outcomes(borough):
    """Outcome distribution over last 12 months."""
    borough_data = raw_df[raw_df["borough"] == borough]
    recent_months = sorted(borough_data["month"].unique())[-12:]
    recent = borough_data[borough_data["month"].isin(recent_months)]
    return recent["outcome_status"].value_counts(normalize=True).head(5)
 

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# DETAILED PREDICTION DISPLAY
# ─────────────────────────────────────────────────────────────────────────────
def show_detailed_prediction(borough, target_date, prediction, months_ahead):
    print("\n" + "=" * 60)
    print(f"  DETAILED PREDICTION REPORT")
    print("=" * 60)
 
    print(f"\n  Borough : {borough}")
    print(f"  Target  : {target_date.strftime('%B %Y')}")
    if months_ahead > 0:
        print(f"  Type    : Future forecast ({months_ahead} months ahead)")
    else:
        print(f"  Type    : Historical period")
 
    # Headline prediction
    print(f"\n  ┌─────────────────────────────────────────┐")
    print(f"  │  PREDICTED TOTAL : {prediction:>10,.0f} crimes  │")
    print(f"  └─────────────────────────────────────────┘")
 
    # Confidence interval
    lower, upper = get_confidence_interval(prediction)
    print(f"\n  95% confidence interval : {lower:,.0f} – {upper:,.0f}")
 
    # ─── CONTEXT ───
    print(f"\n── CONTEXT ──────────────────────────────────────")
    yoy = get_yoy_comparison(borough, target_date)
    if yoy is not None:
        change = (prediction - yoy) / yoy * 100
        direction = "↑" if change > 0 else "↓" if change < 0 else "→"
        print(f"  Same month last year  : {yoy:,.0f} crimes")
        print(f"  Year-on-year change   : {direction} {abs(change):.1f}%")
 
    trend, pct = get_trend(borough)
    print(f"  Recent trend (6mo)    : {trend} ({pct:+.1f}%)")
 
    rank, total = get_borough_rank(prediction, target_date)
    print(f"  Borough rank          : {rank} of {total} boroughs (by predicted volume)")
 
    # ─── CATEGORY BREAKDOWN ───
    print(f"\n── EXPECTED CRIME CATEGORIES ────────────────────")
    print(f"  (based on this borough's last 12 months of patterns)\n")
    breakdown = get_category_breakdown(borough, prediction)
    for category, count in breakdown.head(8).items():
        pct = count / prediction * 100
        bar = "█" * int(pct / 1.5)
        cat_name = category.replace("-", " ").title()
        print(f"  {cat_name:<32} {count:>5,}  {pct:>4.1f}%  {bar}")
 
    # ─── HOTSPOTS ───
    print(f"\n── HISTORICAL HOTSPOTS ──────────────────────────")
    print(f"  (top streets by crime count, last 12 months)\n")
    top_streets = get_top_streets(borough, n=5)
    if len(top_streets) > 0:
        for i, (street, count) in enumerate(top_streets.items(), 1):
            print(f"  {i}. {street:<45} {count:>5,} crimes")
    else:
        print("  No street-level data available for this borough.")
 
    # ─── OUTCOMES ───
    print(f"\n── TYPICAL OUTCOMES ─────────────────────────────")
    print(f"  (how cases were resolved in last 12 months)\n")
    outcomes = get_outcomes(borough)
    for outcome, pct_val in outcomes.items():
        bar = "█" * int(pct_val * 30)
        outcome_short = outcome[:42] if len(outcome) > 42 else outcome
        print(f"  {outcome_short:<43} {pct_val*100:>5.1f}%  {bar}")
 
    print("\n" + "─" * 60)
 
 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO + INTERACTIVE
# ─────────────────────────────────────────────────────────────────────────────
boroughs = sorted(df["borough"].unique())
 
# Auto-demo with Westminster Aug 2026
print("\n\n" + "█" * 60)
print("  AUTO-DEMO: Westminster, August 2026")
print("█" * 60)
 
demo_borough = "Westminster"
demo_target = pd.to_datetime("2026-08-01")
last_date = df["date"].max()
months_ahead = (demo_target.year - last_date.year) * 12 + (demo_target.month - last_date.month)
forecasts = forecast_recursive(model, df, demo_borough, months_ahead)
demo_prediction = forecasts[-1]["prediction"]
show_detailed_prediction(demo_borough, demo_target, demo_prediction, months_ahead)
 
# Interactive
print("\n\n" + "█" * 60)
print("  INTERACTIVE MODE")
print("█" * 60)
print("\nAvailable boroughs:")
for i in range(0, len(boroughs), 3):
    line = ""
    for j in range(3):
        if i + j < len(boroughs):
            line += f"  {i+j+1:>2}. {boroughs[i+j]:<25}"
    print(line)
 
print("\n" + "─" * 60)
print("  Type a borough number and target month for a detailed prediction.")
print("  Type 'q' to quit.")
print("─" * 60)
 
while True:
    print()
    choice = input("Borough number (1-33) or 'q': ").strip().lower()
    if choice == "q":
        print("\nGoodbye! 👋\n")
        break
    try:
        borough = boroughs[int(choice) - 1]
    except (ValueError, IndexError):
        print("  Invalid choice.")
        continue
 
    month_str = input(f"Target month for {borough} (YYYY-MM): ").strip()
    try:
        target_dt = pd.to_datetime(month_str + "-01")
    except (ValueError, TypeError):
        print("  Invalid format. Use YYYY-MM (e.g. 2026-06)")
        continue
 
    last_date_borough = df[df["borough"] == borough]["date"].max()
    earliest = df[df["borough"] == borough]["date"].min()
 
    if target_dt < earliest:
        print(f"  Date too early. Earliest: {earliest.strftime('%Y-%m')}")
        continue
 
    if target_dt <= last_date_borough:
        actual_row = df[(df["borough"] == borough) & (df["date"] == target_dt)]
        if len(actual_row) == 0:
            print("  No data for that month.")
            continue
        pred = float(model.predict(actual_row[FEATURES])[0])
        show_detailed_prediction(borough, target_dt, pred, months_ahead=0)
        actual = float(actual_row["crime_count"].iloc[0])
        print(f"\n  ⓘ Actual value: {actual:,.0f} (model error: {abs(actual-pred):,.0f}, "
              f"{abs(actual-pred)/actual*100:.1f}%)")
    else:
        months_ahead = (target_dt.year - last_date_borough.year) * 12 + \
                       (target_dt.month - last_date_borough.month)
        if months_ahead > 12:
            print(f"  {months_ahead} months ahead is too far for reliable forecasts.")
            continue
        forecasts = forecast_recursive(model, df, borough, months_ahead)
        pred = forecasts[-1]["prediction"]
        show_detailed_prediction(borough, target_dt, pred, months_ahead)
 



████████████████████████████████████████████████████████████
  AUTO-DEMO: Westminster, August 2026
████████████████████████████████████████████████████████████

  DETAILED PREDICTION REPORT

  Borough : Westminster
  Target  : August 2026
  Type    : Future forecast (6 months ahead)

  ┌─────────────────────────────────────────┐
  │  PREDICTED TOTAL :      7,668 crimes  │
  └─────────────────────────────────────────┘

  95% confidence interval : 7,321 – 7,943

── CONTEXT ──────────────────────────────────────
  Same month last year  : 7,925 crimes
  Year-on-year change   : ↓ 3.2%
  Recent trend (6mo)    : decreasing (-12.2%)
  Borough rank          : 1 of 33 boroughs (by predicted volume)

── EXPECTED CRIME CATEGORIES ────────────────────
  (based on this borough's last 12 months of patterns)

  Theft From The Person            1,841  24.0%  ████████████████
  Other Theft                      1,332  17.4%  ███████████
  Anti Social Behaviour            1,128  14.7%  █████████
  Viole

Borough number (1-33) or 'q':  7
Target month for City of London (YYYY-MM):  2027-01



  DETAILED PREDICTION REPORT

  Borough : City of London
  Target  : January 2027
  Type    : Future forecast (11 months ahead)

  ┌─────────────────────────────────────────┐
  │  PREDICTED TOTAL :        547 crimes  │
  └─────────────────────────────────────────┘

  95% confidence interval : 200 – 822

── CONTEXT ──────────────────────────────────────
  Same month last year  : 805 crimes
  Year-on-year change   : ↓ 32.1%
  Recent trend (6mo)    : increasing (+11.1%)
  Borough rank          : 34 of 33 boroughs (by predicted volume)

── EXPECTED CRIME CATEGORIES ────────────────────
  (based on this borough's last 12 months of patterns)

  Other Theft                        130  23.8%  ███████████████
  Violent Crime                       88  16.1%  ██████████
  Shoplifting                         83  15.2%  ██████████
  Theft From The Person               82  15.0%  █████████
  Public Order                        35   6.4%  ████
  Burglary                            28   5.1%  ███
  D